<a href="https://colab.research.google.com/github/Chitlangia-Vedant/Text-Classification-ML/blob/main/Text_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import warnings
warnings.filterwarnings("ignore")

!pip install ydata-profiling

from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn import tree
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn import metrics
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.model_selection import train_test_split

import pandas as pd
from ydata_profiling import ProfileReport

from matplotlib import pyplot as plt
import seaborn as sns
import os
import string
import re

import numpy as np
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [ ]:
import runpy
import time

print(os.listdir('./data_files'))

##### Please select the dataset you want to work on #####

data = 'imdb_data.csv'

path = "download_" + data.replace("_data.csv",".py")
print("\n" + "Working on " + data.replace(".csv",""))
time.sleep(5)

['.ipynb_checkpoints', 'imdb_data.csv']

Working on imdb_data


In [ ]:
print("Pre-text processing")

def text_clean1(text):
    textclean = []
    gibber = ["/", " / ", "#", "*", "&", " in ", " and ", " of ", " for "]
    for i in text:
        i = i.lower()
        # print(i)
        for ch in string.punctuation:
            i = i.replace(ch, " ")
        for x in gibber:
            i = i.replace(x, " ")
        i = i.strip()
        i = " ".join(i.split())
        textclean.append(i)
    return textclean


def text_clean(text):
    textclean = []

    gibber = ["null", " nan "]
    rules = [
        {r'>\s+': u'>'},  # remove spaces after a tag opens or closes
        {r'\s+': u' '},  # replace consecutive spaces
        {r'\s*<br\s*/?>\s*': u'\n'},  # newline after a <br>
        {r'</(div)\s*>\s*': u'\n'},  # newline after </p> and </div> and <h1/>...
        {r'</(p|h\d)\s*>\s*': u'\n\n'},  # newline after </p> and </div> and <h1/>...
        {r'<head>.*<\s*(/head|body)[^>]*>': u''},  # remove <head> to </head>
        {r'<a\s+href="([^"]+)"[^>]*>.*</a>': r'\1'},  # show links instead of texts
        {r'[ \t]*<[^<]*?/?>': u''},  # remove remaining tags
        {r'^\s+': u''}  # remove spaces at the beginning
    ]
    stop_words = set(stopwords.words('english'))
    not_stopwords = {"up", "down", "Up", "Down","not", "no", "very", "too", "never", "nor","n't","cannot","hardly","barely"}
    stop_words = set([word for word in stop_words if word not in not_stopwords])
    regEx = r'(?:\d{1,2}[/]\d{1,2}[/]\d{2,4})|(?:\d{1,2}[-]\d{1,2}[-]\d{2,4})|(?:\d{1,2}[\s]\d{1,2}[\s]\d{2,4})|(?:\d{1,2}[-/th|st|nd|rd\s]*)?(?:Jan+|Feb+|Mar+|Apr+|May+|Jun+|Jul+|Aug+|Sep+|Oct+|Nov+|Dec+)[a-z\s,.]*(?:\d{1,2}[-/th|st|nd|rd)\s,]*)?(?:\d{2,4})'

    for i in text:  # print(i)
        i = i.replace(".00", " ")
        result = re.findall(regEx, i)
        for item in result:
            i = i.replace(item, "")  # Removes dates from text

        i = i.lower()
        # print(i)
        for ch in string.punctuation:  # removes punctuations from all the lines
            if ch == "/" or ch == "-":
                if "-1/" in i or "1/" in i:
                    i = i
                else:
                    i = i.replace(ch, " ")
            else:
                i = i.replace(ch, " ")

        # i = i.strip()
        # i = " ".join(i.split())
        for x in gibber:
            # print(i)
            i = i.replace(x, " ")
        for rule in rules:
            for (k, v) in rule.items():
                regex = re.compile(k)
                i = regex.sub(v, i)

        i = re.sub(r'(\d+)\s+(\d+)\s|(\d{3})\s(\d{3})', r'\1\2', i)

        word_tokens = word_tokenize(i)
        # filtered_sentence = [w for w in word_tokens if not w in stop_words]
        token_sentence = [
            w for w in word_tokens
            if w not in stop_words and len(w) > 2
        ]
        # token_sentence = list(unique_everseen(token_sentence)) #To remove duplicates in the list
        i = " ".join([w for w in token_sentence])
        # i = ''.join([l for l in i if not l.isdigit()]) # Removes all the numerics from string
        i = re.sub(' +', ' ', i)  # removes extra spaces from the entire string
        i = i.strip()
        textclean.append(i)

    return textclean

Pre-text processing


In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
df = pd.read_csv('./data_files/' + data)
text = df['Text'].values.astype(str)
label = df['Label'].values
textclean = text_clean(text)
df['CleanText'] = textclean

df.info()
ProfileReport(df, title="Pandas Profiling Report")

is_train = df['Type']=='train'
df_train = df[is_train]
X_train, y_train = df_train['CleanText'].values.astype(str), df_train['Label'].values
is_test = df['Type']=='test'
df_test = df[is_test]
X_test, y_test = df_test['CleanText'].values.astype(str), df_test['Label'].values

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   ID         50000 non-null  int64 
 1   Rating     50000 non-null  int64 
 2   Text       50000 non-null  object
 3   Label      50000 non-null  int64 
 4   Type       50000 non-null  object
 5   CleanText  50000 non-null  object
dtypes: int64(3), object(3)
memory usage: 2.3+ MB


Linear SVC

In [ ]:
svc_mod = Pipeline([('vect', CountVectorizer()),
                     ('tfidf', TfidfTransformer()),
                     ('clf', LinearSVC()),
                     ])

svc_mod.fit(X_train, y_train)
predicted = svc_mod.predict(X_test)

# Analyze model metrics
print("Linear SVC Classification Report")
print(metrics.classification_report(y_test, predicted))

print("Linear SVC Confusion Matrix")
print(metrics.confusion_matrix(y_test, predicted))

Linear SVC Classification Report
              precision    recall  f1-score   support

           0       0.86      0.88      0.87     12500
           1       0.88      0.86      0.87     12500

    accuracy                           0.87     25000
   macro avg       0.87      0.87      0.87     25000
weighted avg       0.87      0.87      0.87     25000

Linear SVC Confusion Matrix
[[11055  1445]
 [ 1745 10755]]


Random Forest

In [ ]:
RF_mod = Pipeline([('vect', CountVectorizer()),
                     ('tfidf', TfidfTransformer()),
                     ('clf', RandomForestClassifier(n_estimators=100)),
                     ])

RF_mod.fit(X_train, y_train)
predicted = RF_mod.predict(X_test)

# Analyze model metrics
print("Random Forest Classification Report")
print(metrics.classification_report(y_test, predicted))

print("Random Forest Confusion Matrix")
print(metrics.confusion_matrix(y_test, predicted))

Random Forest Classification Report
              precision    recall  f1-score   support

           0       0.85      0.86      0.85     12500
           1       0.86      0.84      0.85     12500

    accuracy                           0.85     25000
   macro avg       0.85      0.85      0.85     25000
weighted avg       0.85      0.85      0.85     25000

Random Forest Confusion Matrix
[[10759  1741]
 [ 1966 10534]]


KNN

In [ ]:
knn_mod = Pipeline([('vect', CountVectorizer()),
                     ('tfidf', TfidfTransformer()),
                     ('clf', KNeighborsClassifier()),
                     ])

knn_mod.fit(X_train, y_train)
predicted = knn_mod.predict(X_test)

# Analyze model metrics
print("KNN Classification Report")
print(metrics.classification_report(y_test, predicted))

print("KNN Confusion Matrix")
print(metrics.confusion_matrix(y_test, predicted))

KNN Classification Report
              precision    recall  f1-score   support

           0       0.65      0.67      0.66     12500
           1       0.66      0.64      0.65     12500

    accuracy                           0.65     25000
   macro avg       0.65      0.65      0.65     25000
weighted avg       0.65      0.65      0.65     25000

KNN Confusion Matrix
[[8374 4126]
 [4558 7942]]


*Decision* Tree

In [ ]:
DT_mod = Pipeline([('vect', CountVectorizer()),
                     ('tfidf', TfidfTransformer()),
                     ('clf', tree.DecisionTreeClassifier()),
                     ])

DT_mod.fit(X_train, y_train)
predicted = DT_mod.predict(X_test)

# Analyze model metrics
print("Decision Tree Classification Report")
print(metrics.classification_report(y_test, predicted))

print("Decision Tree Confusion Matrix")
print(metrics.confusion_matrix(y_test, predicted))

Decision Tree Classification Report
              precision    recall  f1-score   support

           0       0.71      0.73      0.72     12500
           1       0.72      0.71      0.72     12500

    accuracy                           0.72     25000
   macro avg       0.72      0.72      0.72     25000
weighted avg       0.72      0.72      0.72     25000

Decision Tree Confusion Matrix
[[9065 3435]
 [3631 8869]]


Multinomial Naive Bayes

In [ ]:
MNB_mod = Pipeline([('vect', CountVectorizer()),
                     ('tfidf', TfidfTransformer()),
                     ('clf', MultinomialNB()),
                     ])

MNB_mod.fit(X_train, y_train)
predicted = MNB_mod.predict(X_test)

# Analyze model metrics
print("Multinomial Naive Bayes Classification Report")
print(metrics.classification_report(y_test, predicted))

print("Multinomial Naive Bayes Confusion Matrix")
print(metrics.confusion_matrix(y_test, predicted))

Multinomial Naive Bayes Classification Report
              precision    recall  f1-score   support

           0       0.81      0.88      0.84     12500
           1       0.87      0.79      0.83     12500

    accuracy                           0.84     25000
   macro avg       0.84      0.84      0.84     25000
weighted avg       0.84      0.84      0.84     25000

Multinomial Naive Bayes Confusion Matrix
[[11018  1482]
 [ 2612  9888]]


Improved Linear SVC

In [ ]:
isvc_mod = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1,3),
        min_df=2,
        max_df=0.9,
        max_features=80000,
        sublinear_tf=True
    )),
    ('clf', LinearSVC(
        C=2,
        loss="squared_hinge",
        class_weight='balanced'
    ))
])

isvc_mod.fit(X_train, y_train)
predicted = isvc_mod.predict(X_test)

# Analyze model metrics
print("Improved Linear SVC Classification Report")
print(metrics.classification_report(y_test, predicted))

print("Improved Linear SVC Confusion Matrix")
print(metrics.confusion_matrix(y_test, predicted))

Improved Linear SVC Classification Report
              precision    recall  f1-score   support

           0       0.88      0.90      0.89     12500
           1       0.90      0.88      0.89     12500

    accuracy                           0.89     25000
   macro avg       0.89      0.89      0.89     25000
weighted avg       0.89      0.89      0.89     25000

Improved Linear SVC Confusion Matrix
[[11225  1275]
 [ 1497 11003]]


GUI

In [ ]:
import pickle
import gradio as gr

!pip install gradio
pickle.dump(isvc_mod, open("model.pkl", "wb"))